# Does CPT → SFT beat SFT → CPT?

Two-stage domain adaptation of `unsloth/SmolLM-135M` on SEC 10-K financial text,
run in both orders and compared.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s1mran/finetuning/blob/main/run_in_colab.ipynb)

|  | CPT | SFT |
|---|---|---|
| objective | next-token, loss on **every** token | Alpaca pairs, loss on **response span only** |
| LoRA rank | 32 | 16 |
| targets | linear + `embed_tokens` + `lm_head` | linear only |
| packing | on | off (masking needs 1 example/sequence) |

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**.
Unsloth is CUDA-only; there is no CPU or Apple-MPS fallback.

Run the cells in order. Cells 1–3 are setup and must be re-run after any
runtime restart or VM recycle — Colab wipes installed packages every time.

## 1. Confirm the GPU is actually attached

This must print a GPU table. If it errors, you are on a CPU runtime and
everything below will fail at `import unsloth` — go fix the runtime type first.

In [ ]:
!nvidia-smi

## 2. Install dependencies

~2 minutes. Needs re-running after every restart; Colab does not persist
`dist-packages`. No kernel restart required afterwards, because the training
scripts run as subprocesses (`!python`) that pick up the new packages.

In [ ]:
!pip install -q unsloth trl peft transformers datasets accelerate bitsandbytes

## 3. Get the code

Clones on a fresh VM, pulls if the directory already exists.

In [ ]:
!git clone https://github.com/s1mran/finetuning.git /content/finetuning 2>/dev/null || git -C /content/finetuning pull
!ls -la /content/finetuning

## 4. Smoke test — 30 steps per stage

Checks the whole pipeline end to end in ~2 minutes before you commit to a full
run. **Three things to verify in the output:**

1. `[cpt-data] general replay: Salesforce/wikitext, N chunks` — if this says
   *unavailable*, the general-replay mix is off and the script will now stop
   rather than silently report a fake perplexity.
2. `[ppl] base -> {'domain': ~22, 'general': <a real number>}` — a general
   perplexity of exactly `1.0` means an empty corpus, not a perfect model.
3. `[mask] supervising N/M tokens` — response-only masking is live. This should
   be a *minority* of tokens; if it were ~100% the prompt masking silently
   failed.

Everything runs from `/content/finetuning` so the `runs/` output directory
lands inside the repo.

In [ ]:
!cd /content/finetuning && python cpt_then_sft.py --smoke

## 5. Full run — CPT → SFT

~15 minutes on a T4. This is the ordering the README argues for: CPT teaches
the model to *sound* like the domain, SFT then teaches it to *answer*, leaving
instruction-following as the freshest thing in the weights.

In [ ]:
!cd /content/finetuning && python cpt_then_sft.py

## 6. Full run — SFT → CPT (the ablation)

Same seed, same step counts, stages reversed. Use matching step counts or the
comparison means nothing.

Expected: domain perplexity roughly comparable, but the alpaca probes ramble
past the answer and never emit EOS — all-token loss on raw prose has no reason
to preserve response-only behaviour. `eos_rate` in the report measures exactly
that.

In [ ]:
!cd /content/finetuning && python sft_then_cpt.py

## 7. Compare the two runs

Reads both `report.json` files and lays the numbers side by side. The
perplexity columns should be comparable; the probe behaviour should not be.

In [ ]:
import json, math
from pathlib import Path

RUNS = Path("/content/finetuning/runs")

# (report dir, stage-1 ppl key, stage-2 ppl key, stage-1 probe key)
LAYOUT = {
    "CPT -> SFT": ("cpt_then_sft", "ppl_after_cpt", "ppl_after_sft", "probes_after_cpt"),
    "SFT -> CPT": ("sft_then_cpt", "ppl_after_sft", "ppl_after_cpt", "probes_after_sft"),
}

def num(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "  n/a "
    return f"{v:6.2f}"

reports = {}
for name, (d, *_rest) in LAYOUT.items():
    p = RUNS / d / "report.json"
    if p.exists():
        reports[name] = json.loads(p.read_text())
    else:
        print(f"missing {p} -- run that cell first")

for name, rep in reports.items():
    _, s1, s2, s1_probe = LAYOUT[name]
    print("=" * 66)
    print(f"{name}   (stage 1 -> stage 2)")
    print("=" * 66)
    for dom in ("domain", "general"):
        b = rep.get("ppl_before", {}).get(dom)
        a = rep.get(s1, {}).get(dom)
        c = rep.get(s2, {}).get(dom)
        print(f"  {dom:>7} ppl : {num(b)}  ->{num(a)}  ->{num(c)}")
    for stage, key in (("base", "probes_base"),
                       ("stage1", s1_probe),
                       ("final", "probes_final")):
        rate = (rep.get(key) or {}).get("eos_rate")
        if rate is not None:
            print(f"  {stage:>7} eos : {rate:.0%}")
    print()

# The qualitative half: does the final model answer *and stop*?
if reports:
    print("=" * 66)
    print("FINAL ALPACA PROBES")
    print("=" * 66)
    probe_keys = {k for r in reports.values()
                  for k in (r.get("probes_final") or {}) if k.startswith("alpaca::")}
    for pk in sorted(probe_keys):
        print(f"\n  {pk.split('::', 1)[1]}")
        for name, rep in reports.items():
            ans = (rep.get("probes_final") or {}).get(pk, "<missing>")
            print(f"    [{name}] {ans[:240]}")

## 8. Keep the results (optional)

Colab recycles VMs on idle and caps session length, so anything left in
`/content` is temporary. Two ways to keep the artifacts:

**Google Drive** — copies adapters, merged model and reports.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/finetuning_runs"
!cp -r /content/finetuning/runs/* "/content/drive/MyDrive/finetuning_runs/"
!ls -R "/content/drive/MyDrive/finetuning_runs" | head -40

**Hugging Face Hub** — pushes the merged model plus a generated model card.
Needs a **write**-scoped token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Store it in Colab's secrets pane (🔑 in the left sidebar) as `HF_TOKEN`, then
run the cell below. The bare `--push-to-hub` flag publishes to
`sidhusarkar/<run name>`; pass your own username to override.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!cd /content/finetuning && python cpt_then_sft.py --push-to-hub

## Known limitation

`eloukas/edgar-corpus` (the raw 10-K MD&A prose intended for CPT) is
script-based, and `datasets` dropped support for script loaders. Every EDGAR
mirror checked so far has the same problem, so CPT currently falls back to the
`context` column of `virattt/financial-qa-10K` — **the same corpus stage 2
trains on**.

The run works and the numbers are real, but CPT and SFT seeing identical text
weakens the ordering claim. Worth resolving before drawing strong conclusions.